# INPE: processamento e visualizacao individual

Este notebook usa a base INPE previamente validada, aplica explicitamente o periodo definido para a fonte e gera somente a figura historica individual.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180

def encontrar_pasta_2026(inicio=Path.cwd()):
    inicio = inicio.resolve()
    for pasta in (inicio, *inicio.parents):
        candidata = pasta / '2026'
        if candidata.exists():
            return candidata
    raise FileNotFoundError('Nao foi encontrada a pasta 2026 a partir do diretorio atual.')

PASTA_2026 = encontrar_pasta_2026()
PASTA_SAIDA = PASTA_2026 / 'visualizacoes_comparativas_bases'
PASTA_FONTE = PASTA_SAIDA / 'figuras_por_fonte'
PASTA_FONTE.mkdir(parents=True, exist_ok=True)

FREQUENCIA_VISUAL = 'D'

ROTULOS = {
    'radiacao_onda_curta': 'Radiacao de onda curta (W/m2)',
    'temperatura': 'Temperatura (C)',
    'umidade_relativa': 'Umidade relativa (%)',
}

def recortar_periodo(dados, inicio, fim):
    return dados.loc[
        (dados['data_hora'] >= inicio) & (dados['data_hora'] <= fim)
    ].copy().reset_index(drop=True)

def serie_visual(dados, coluna):
    return dados.set_index('data_hora')[coluna].resample(FREQUENCIA_VISUAL).mean()

def plotar_evolucao(dados, nome, colunas, arquivo, ncol=1):
    colunas = [coluna for coluna in colunas if coluna in dados.columns]
    nlin = int(np.ceil(len(colunas) / ncol))
    fig, eixos = plt.subplots(nlin, ncol, figsize=(14, 3.6 * nlin), sharex=True, squeeze=False)
    for eixo, coluna in zip(eixos.flat, colunas):
        serie_visual(dados, coluna).plot(ax=eixo, color='#1f5a85', linewidth=0.8)
        eixo.set_title(ROTULOS.get(coluna, coluna))
        eixo.set_xlabel('')
        eixo.set_ylabel('')
    for eixo in eixos.flat[len(colunas):]:
        eixo.remove()
    inicio = dados['data_hora'].min().strftime('%d/%m/%Y')
    fim = dados['data_hora'].max().strftime('%d/%m/%Y')
    fig.suptitle(f'Evolucao temporal - {nome} ({inicio} a {fim}; media diaria)', y=1.01)
    fig.tight_layout()
    fig.savefig(arquivo, bbox_inches='tight')
    plt.show()
    plt.close(fig)

ARQUIVO_INPE = PASTA_SAIDA / 'dados_padronizados' / 'inpe_2024-08-28_11h_continua_umidade_tratada.csv'
INICIO_INPE = pd.Timestamp('2024-08-28 11:00:00')
FIM_INPE = pd.Timestamp('2025-02-11 09:00:00')
INVALIDOS = [-9999, -999, -6999, 7999, 9999]

def ler_inpe():
    if not ARQUIVO_INPE.exists():
        raise FileNotFoundError(
            f'Base INPE tratada nao encontrada: {ARQUIVO_INPE}. Execute primeiro validar_umidade_inpe.ipynb.'
        )
    dados = pd.read_csv(ARQUIVO_INPE, parse_dates=['data_hora'])
    for coluna in ['radiacao_onda_curta', 'temperatura', 'umidade_relativa']:
        dados[coluna] = pd.to_numeric(dados[coluna], errors='coerce').replace(INVALIDOS, np.nan)
    dados = dados.dropna(subset=['data_hora']).sort_values('data_hora').drop_duplicates('data_hora')
    return recortar_periodo(dados, INICIO_INPE, FIM_INPE)

inpe = ler_inpe()
print(f'INPE: {len(inpe)} registros | {inpe.data_hora.min()} a {inpe.data_hora.max()}')
display(inpe.describe(include='all'))


In [ ]:
plotar_evolucao(inpe, 'INPE', ['radiacao_onda_curta', 'temperatura', 'umidade_relativa'], PASTA_FONTE / 'inpe_evolucao.png')
